In [16]:
#Esercizio 1. Tipo di problema e preparazione dei dati

#Dati: DataFrame clienti di almeno 200 righe generato con NumPy (np.random.seed(42)): 
#mesi, reclami, spesa (numeriche), contratto (Mensile, Annuale, Biennale), churn (0/1, circa 15% di valori 1).

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

#1. classificare tre casi (fatturato del mese successivo, reso di un ordine si/no, 
#gruppi di negozi simili) come regressione, classificazione o clustering, indicando la colonna target.

np.random.seed(42)
n = 200

df = pd.DataFrame({
    "mesi": np.random.randint(1, 48, n),
    "reclami": np.random.randint(0, 5, n),
    "spesa": np.random.uniform(20, 500, n).round(2),
    "contratto": np.random.choice(["Mensile", "Annuale", "Biennale"], n),
    "churn": np.random.choice([0, 1], n, p=[0.85, 0.15])
})

print(clienti)

     mesi  reclami       spesa contratto
0      52        0   90.790122   Annuale
1      15        0   67.069285   Annuale
2      72        2   72.496991  Biennale
3      61        2  128.499043   Mensile
4      21        1   61.790656   Annuale
..    ...      ...         ...       ...
195    33        2   38.200986   Mensile
196    40        0   57.406648   Annuale
197    39        0   49.818697   Mensile
198     1        0  109.273555   Mensile
199    11        0   24.023892   Mensile

[200 rows x 4 columns]


In [9]:
#2. applicare pd.get_dummies() alla colonna contratto con drop_first=True e dtype=int, 
#poi separare X (feature) e y (churn).

df = pd.DataFrame({
    "mesi": np.random.randint(1, 48, n),
    "reclami": np.random.randint(0, 5, n),
    "spesa": np.random.uniform(20, 500, n).round(2),
    "contratto": np.random.choice(["Mensile", "Annuale", "Biennale"], n),
    "churn": np.random.choice([0, 1], n, p=[0.85, 0.15])
})

df_dummies = pd.get_dummies(df, columns=["contratto"], drop_first=True, dtype=int)
#trasforma le colonne in colonne binarie 0/1 per ciascuna categoria, qui contratto
#elimina la prima categoria

X = df_dummies.drop(columns=["churn"])
y = df_dummies["churn"]


In [12]:
#3. dividere i dati con train_test_split(test_size=0.2, stratify=y, random_state=42) 
#e verificare con y_train.mean() e y_test.mean() che la quota di churn coincida.


X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print(y_train.mean())
print(y_test.mean())


0.15625
0.15


In [15]:
#4. standardizzare le feature con StandardScaler: fit_transform() su X_train, transform() su X_test.

scaler = StandardScaler()

X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

#Definizione richiesta: scrivere la definizione di data leakage e spiegare perché lo scaler non va 
#addestrato sull'intero dataset.

#Un data lakage è è un errore in cui le informazioni del test finiscono in quelle del modello di training
# In pratica il modello ha delle informazioni che non dovrebbe avere che falsando poi il risultato

#Lo Scaler non va addestrato sull'intero dataset in quanto portebbe appunto al data lakage 
#se calcoliamo media e deviazione nel test questi finirebbero poi nei dati di traning gonfiando le prestazioni

#Criterio di completamento: tre casi classificati con target indicato, due colonne dummy create, 
#quote di churn in train e test uguali al centesimo, X_train_s con media O e deviazione standard 1 per ogni colonna.

In [21]:
#Esercizio 2. Regressione logistica per il churn

#Dati: X_train_s, X_test_s, y_train e y_test prodotti nell'esercizio 1.

#1. con il modello z = - 1,0 + 0,6 reclami - 0,04 mesi calcolare a mano z e p 
#per un cliente con 4 reclami e 12 mesi, poi verificare il risultato in Python con una funzione sigmoide.

#z = -1,0 + 0,6 * reclami - 0,04 × mesi
#z = -1,0 + 0,6 * 4 - 0,04 × 12
#z = -1,0 + 2,4 - 0,48
#z = 0,92

#p = 1 / (1 + e^(-z))
#p = 1 / (1 + e^(-0,92))
#p ≈ 0,715, ovvero 0,72

reclami = 4
mesi = 12

z = -1.0 + 0.6 * reclami - 0.04 * mesi
print(z) 

def sigmoide(z):
    return 1 / (1 + np.exp(-z))

p = sigmoide(z)
print(p)


0.9199999999999999
0.7150421057009897


In [22]:
#2. addestrare LogisticRegression(max_iter=1000) con fit() su X_train_s e y_train 
#e ottenere le classi previste con predict() su X_test_s.

model = LogisticRegression(max_iter=1000)
model.fit(X_train_s, y_train)

y_pred = model.predict(X_test_s)

In [23]:
#3. ottenere le probabilità di churn con predict_proba(X_test_s)[:, 1] 
#e contare i clienti classificati churn con soglia 0,5 e con soglia 0,3.

proba_churn = model.predict_proba(X_test_s)[:, 1]

n_churn_05 = (proba_churn >= 0.5).sum()
n_churn_03 = (proba_churn >= 0.3).sum()

print(n_churn_05)
print(n_churn_03)

0
0


In [24]:
#4. costruire la Series dei coefficienti con index=X.columns, 
#calcolare gli odds ratio con np.exp() e ordinarli con sort_values ().

coefficienti = pd.Series(model.coef_[0], index=X.columns)

odds_ratio = np.exp(coefficienti)

odds_ratio_ordinati = odds_ratio.sort_values(ascending=False)
print(odds_ratio_ordinati)

#Definizione richiesta: scrivere la definizione di odds e di odds ratio 
#e spiegare perché la regressione logistica è un algoritmo di classificazione.

#Gli odds esprimono il rapporto tra la probabilità che un evento accada e la probabilità che non accada
#L’odds ratio invece indica di quanto cambiano gli odds per un aumento di una unità in una variabile indipendente, 
#mantenendo le altre però costanti.

#la regressione logistica predice una probabilità tra 0 e 1 che un’osservazione appartenga a una determinata classe, 
#grazie alla funzione sigmoide

#Criterio di completamento: z = 0,92 e p = 0,72 (churn = 1) confermati in Python, 
#conteggi dei churn previsti con le due soglie,feature con l'odds ratio più alto indicata nella cella Markdown.

mesi                  1.257773
contratto_Biennale    1.249002
contratto_Mensile     1.214021
reclami               0.944164
spesa                 0.825378
dtype: float64
